# Clase 7 — LLMs sin magia: conceptos clave antes de la práctica

> En esta clase no vamos a entrenar un modelo ni a meternos con mucho código. La meta es entender, con ejemplos simples, qué idea hace posible a un LLM.

## Lo que te vas a llevar

| Sección | Idea principal |
|---|---|
| 1 | Un LLM aprende a predecir el siguiente token |
| 2 | Texto, tokens y vocabulario |
| 3 | Contexto y ventana de atención |
| 4 | Probabilidades y temperatura |
| 5 | Qué hace un Transformer, sin bajar al detalle matemático |
| 6 | Por qué llama.cpp y la cuantización son importantes |

> Idea-fuerza: dado un contexto, el modelo calcula cuál podría ser el próximo token. Después repite el proceso una y otra vez.

---
## 1. Preparación mínima

Vamos a usar solamente Python estándar. Eso nos permite enfocarnos en las ideas y no en las librerías.

La propuesta para esta hora es simple:

1. Entender qué problema resuelve un LLM.
2. Ver cómo se parte un texto en tokens.
3. Observar qué significa contexto.
4. Mirar cómo cambian las probabilidades con la temperatura.
5. Cerrar conectando todo con llama.cpp.

In [8]:
from collections import Counter, defaultdict
import math
import random

random.seed(7)
print("Entorno listo: usaremos ejemplos simples y código corto.")

Entorno listo: usaremos ejemplos simples y código corto.


---
## 2. La idea central: predecir qué viene después

Cuando vos leés la frase "el mate se toma con...", seguramente imaginás varias continuaciones posibles. Un LLM hace algo parecido, pero usando números y probabilidades.

El ciclo mental que queremos entender hoy es este:

1. El texto se convierte en tokens.
2. El modelo mira el contexto disponible.
3. Calcula probabilidades para el siguiente token.
4. Elige uno y repite.

No piensa como una persona. Detecta patrones en grandes cantidades de texto y los reutiliza al generar una respuesta.

---
## 3. Texto a tokens: tres formas de partir una frase

Los modelos no trabajan con frases enteras. Trabajan con piezas.

Las tres estrategias más conocidas son:

- Character-level: un token por carácter.
- Word-level: un token por palabra.
- Subword: un token por fragmento útil de palabra.

Los LLM modernos suelen usar `subwords` porque equilibran mejor tamaño de vocabulario y longitud de secuencia.

In [9]:
def tokenizar_caracteres(texto):
    return list(texto)

def tokenizar_palabras(texto):
    texto = texto.replace(",", " ,").replace(".", " .")
    return texto.split()

def dividir_en_subwords_toy(palabra, vocabulario):
    tokens = []
    i = 0
    while i < len(palabra):
        encontrado = None
        for j in range(len(palabra), i, -1):
            pieza = palabra[i:j]
            if pieza in vocabulario:
                encontrado = pieza
                break
        if encontrado is None:
            encontrado = palabra[i]
        tokens.append(encontrado)
        i += len(encontrado)
    return tokens

def tokenizar_subwords_toy(texto, vocabulario):
    resultado = []
    for palabra in texto.lower().split():
        resultado.extend(dividir_en_subwords_toy(palabra, vocabulario))
        resultado.append("▁")
    return resultado[:-1]

frase = "inteligencia artificial aprende rapido"
vocabulario_subwords = {"intelig", "encia", "art", "ificial", "aprende", "ra", "pido"}

tokens_char = tokenizar_caracteres(frase)
tokens_word = tokenizar_palabras(frase)
tokens_sub = tokenizar_subwords_toy(frase, vocabulario_subwords)

print("Frase original:", frase)
print()
print("1) Character-level")
print(tokens_char)
print("Cantidad de tokens:", len(tokens_char))
print()
print("2) Word-level")
print(tokens_word)
print("Cantidad de tokens:", len(tokens_word))
print()
print("3) Subwords (ejemplo simplificado)")
print(tokens_sub)
print("Cantidad de tokens:", len(tokens_sub))

Frase original: inteligencia artificial aprende rapido

1) Character-level
['i', 'n', 't', 'e', 'l', 'i', 'g', 'e', 'n', 'c', 'i', 'a', ' ', 'a', 'r', 't', 'i', 'f', 'i', 'c', 'i', 'a', 'l', ' ', 'a', 'p', 'r', 'e', 'n', 'd', 'e', ' ', 'r', 'a', 'p', 'i', 'd', 'o']
Cantidad de tokens: 38

2) Word-level
['inteligencia', 'artificial', 'aprende', 'rapido']
Cantidad de tokens: 4

3) Subwords (ejemplo simplificado)
['intelig', 'encia', '▁', 'art', 'ificial', '▁', 'aprende', '▁', 'ra', 'pido']
Cantidad de tokens: 10


> Lo importante no es memorizar el código. Lo importante es notar la diferencia de cantidad. Con caracteres hay más pasos. Con palabras hay menos pasos, pero aparecen problemas con palabras nuevas. Subword intenta quedarse con lo mejor de ambos mundos.

In [10]:
mini_corpus = [
    "la inteligencia artificial aprende de datos",
    "la inteligencia artificial reconoce patrones",
    "un modelo grande necesita muchos datos"
]

vocab_chars = sorted(set(" \n".join(mini_corpus)))
vocab_words = sorted({palabra for texto in mini_corpus for palabra in texto.split()})

print("Vocabulario character-level:")
print(vocab_chars)
print("Tamaño:", len(vocab_chars))
print()
print("Vocabulario word-level:")
print(vocab_words)
print("Tamaño:", len(vocab_words))

Vocabulario character-level:
['\n', ' ', 'a', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u']
Tamaño: 19

Vocabulario word-level:
['aprende', 'artificial', 'datos', 'de', 'grande', 'inteligencia', 'la', 'modelo', 'muchos', 'necesita', 'patrones', 'reconoce', 'un']
Tamaño: 13


---
## 4. Contexto: el fragmento de texto que el modelo puede mirar

Un modelo no mira todo el universo al mismo tiempo. Mira una ventana de contexto.

Ejemplo humano:

- Si leés "la capital de Argentina es", el contexto ayuda muchísimo.
- Si solo ves "es", no alcanza para anticipar casi nada útil.

El tamaño de contexto define cuánto pasado puede usar el modelo para decidir el próximo token.

In [11]:
frase = "la IA aprende del contexto y luego genera texto"
tokens = tokenizar_palabras(frase)
tam_contexto = 4

print("Tokens:", tokens)
print()
print("Pares contexto -> siguiente token")
for i in range(1, len(tokens)):
    inicio = max(0, i - tam_contexto)
    contexto = tokens[inicio:i]
    objetivo = tokens[i]
    print(f"{contexto} -> {objetivo}")

Tokens: ['la', 'IA', 'aprende', 'del', 'contexto', 'y', 'luego', 'genera', 'texto']

Pares contexto -> siguiente token
['la'] -> IA
['la', 'IA'] -> aprende
['la', 'IA', 'aprende'] -> del
['la', 'IA', 'aprende', 'del'] -> contexto
['IA', 'aprende', 'del', 'contexto'] -> y
['aprende', 'del', 'contexto', 'y'] -> luego
['del', 'contexto', 'y', 'luego'] -> genera
['contexto', 'y', 'luego', 'genera'] -> texto


Ese formato contexto -> siguiente token es, conceptualmente, el corazón del entrenamiento de un LLM.

---
## 5. Probabilidades: el modelo no responde con una sola opción fija

Un LLM no guarda una respuesta exacta para cada frase. Lo que produce es una distribución de probabilidades.

El siguiente ejemplo no es un LLM real. Es una maqueta chiquita para entender la idea.

In [12]:
corpus = [
    "la ia aprende de ejemplos",
    "la ia aprende del contexto",
    "la ia aprende con practica",
    "el modelo predice el siguiente token"
]

def construir_tabla_siguiente_token(textos):
    tabla = defaultdict(Counter)
    for texto in textos:
        tokens = tokenizar_palabras(texto.lower())
        for actual, siguiente in zip(tokens, tokens[1:]):
            tabla[actual][siguiente] += 1
    return tabla

tabla = construir_tabla_siguiente_token(corpus)
token_actual = "aprende"
conteos = tabla[token_actual]
total = sum(conteos.values())

print(f"Si el token actual es '{token_actual}', estas son las continuaciones vistas en el corpus:")
for token, cantidad in conteos.most_common():
    prob = cantidad / total
    print(f"- {token:10s} -> {prob:.2%}")

Si el token actual es 'aprende', estas son las continuaciones vistas en el corpus:
- de         -> 33.33%
- del        -> 33.33%
- con        -> 33.33%


---
## 6. Temperatura: cómo controlar lo conservador o creativo que será el modelo

La temperatura no le agrega conocimiento al modelo. Solo cambia cómo usa las probabilidades que ya tiene.

- Temperatura baja: favorece mucho a las opciones más probables.
- Temperatura media: equilibrio.
- Temperatura alta: reparte más probabilidad entre opciones menos probables.

In [14]:
logits = {
    "agua": 3.2,
    "yerba": 2.4,
    "hielo": 1.2
}

def softmax_con_temperatura(logits_dict, temperatura):
    items = list(logits_dict.items())
    escalados = [(token, valor / temperatura) for token, valor in items]
    maximo = max(valor for _, valor in escalados)
    exp_vals = [(token, math.exp(valor - maximo)) for token, valor in escalados]
    total = sum(valor for _, valor in exp_vals)
    return {token: valor / total for token, valor in exp_vals}

print("Complea la frase: el mate se toma con...")

for temperatura in [0.5, 1.0, 1.8]:
    print(f"Temperatura = {temperatura}")
    distribucion = softmax_con_temperatura(logits, temperatura)
    for token, prob in distribucion.items():
        print(f"  {token:6s} -> {prob:.2%}")
    print()

Complea la frase: el mate se toma con...
Temperatura = 0.5
  agua   -> 81.95%
  yerba  -> 16.55%
  hielo  -> 1.50%

Temperatura = 1.0
  agua   -> 63.10%
  yerba  -> 28.35%
  hielo  -> 8.54%

Temperatura = 1.8
  agua   -> 50.75%
  yerba  -> 32.54%
  hielo  -> 16.71%



---
## 7. Entonces, ¿qué hace un Transformer?

Sin entrar en detalle técnico, un Transformer hace estas tareas:

| Pieza | Explicación simple |
|---|---|
| Embeddings | Convierte tokens en vectores numéricos |
| Positional embeddings | Le dice al modelo en qué orden aparecen los tokens |
| Attention | Decide qué partes del contexto conviene mirar más |
| Feed-forward | Procesa la información de cada posición |
| Capa de salida | Devuelve probabilidades para el siguiente token |

La atención fue una gran innovación porque permite relacionar palabras lejanas dentro del contexto.

---
## 8. Cuantización y llama.cpp

Un modelo grande puede tener millones o miles de millones de parámetros. Si cada parámetro ocupa muchos bits, el archivo pesa más y hace falta más memoria RAM para usarlo.

La cuantización reduce esa precisión numérica para que el modelo sea más liviano. llama.cpp aprovecha esa idea para ejecutar modelos GGUF en una compu común.

In [15]:
def estimar_memoria_gb(parametros_millones, bits_por_parametro):
    parametros = parametros_millones * 1_000_000
    bytes_totales = parametros * bits_por_parametro / 8
    return bytes_totales / (1024 ** 3)

modelos = [(500, "modelo de 0.5B"), (7000, "modelo de 7B")]
formatos = [(16, "FP16"), (8, "INT8"), (4, "INT4")]

for parametros, nombre in modelos:
    print(nombre)
    for bits, etiqueta in formatos:
        memoria = estimar_memoria_gb(parametros, bits)
        print(f"  {etiqueta:4s} -> {memoria:.2f} GB aprox.")
    print()

modelo de 0.5B
  FP16 -> 0.93 GB aprox.
  INT8 -> 0.47 GB aprox.
  INT4 -> 0.23 GB aprox.

modelo de 7B
  FP16 -> 13.04 GB aprox.
  INT8 -> 6.52 GB aprox.
  INT4 -> 3.26 GB aprox.



---
## 9. Cierre y preguntas guía

Si esta clase salió bien, al terminar deberías poder explicar con tus palabras estas tres ideas:

1. Un LLM aprende patrones para predecir el siguiente token.
2. La tokenización decide cómo se parte el texto y eso cambia mucho el comportamiento del modelo.
3. llama.cpp permite correr modelos cuantizados de forma local, sin depender siempre de una API.

### Actividad breve

- ¿Por qué un modelo con contexto más grande suele responder mejor?
- ¿Qué ventaja práctica tiene usar subwords en vez de caracteres?
- ¿Qué ganamos y qué perdemos al cuantizar un modelo?

> En la clase siguiente vamos a correr un LLM local con llama.cpp y a experimentar con prompts, temperatura y longitud de respuesta.